# Confounder, mediator, collider: what adjusting does

Companion to the course pages *Causal diagrams* and *How to: draw and use a causal diagram*.

We simulate three small worlds, each with a known truth, and estimate the effect of `x` on `y` twice: **without** and **with** adjusting for a third variable. Which estimate is right depends entirely on the variable's role in the diagram, not on how well the model fits.

```
Confounder world:   z -> x,  z -> y,  x -> y     (z is a common cause)
Mediator world:     x -> m -> y,  x -> y         (m is on the path from x to y)
Collider world:     x -> y,  x -> c,  y -> c     (c is a consequence of both)
```

Change the parameters in the next cell and use *Run All*. Needs `numpy`, `pandas`, `matplotlib`.

In [ ]:
# ---- Parameters: change these and re-run ----
DIRECT_EFFECT = 2.0     # direct effect of x on y in every world
STRENGTH = 1.0          # how strongly the third variable is tied to x (and to y, in the confounder world)
N = 2000                # sample size
N_SIMS = 300            # simulated datasets per world
SEED = 0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(SEED)


def coef_of_x(y, x, *others):
    """Least-squares coefficient on x in y ~ x + others."""
    design = np.column_stack([np.ones_like(x), x, *others])
    return np.linalg.lstsq(design, y, rcond=None)[0][1]


def confounder_world(n):
    z = rng.normal(size=n)
    x = STRENGTH * z + rng.normal(size=n)
    y = DIRECT_EFFECT * x + STRENGTH * z + rng.normal(size=n)
    return y, x, z


def mediator_world(n):
    x = rng.normal(size=n)
    m = STRENGTH * x + rng.normal(size=n)
    y = DIRECT_EFFECT * x + STRENGTH * m + rng.normal(size=n)
    return y, x, m


def collider_world(n):
    x = rng.normal(size=n)
    y = DIRECT_EFFECT * x + rng.normal(size=n)
    c = STRENGTH * x + STRENGTH * y + rng.normal(size=n)
    return y, x, c


# The effect we want in each world is the TOTAL effect of x on y.
worlds = {
    "confounder (z)": (confounder_world, DIRECT_EFFECT),
    "mediator (m)": (mediator_world, DIRECT_EFFECT + STRENGTH * STRENGTH),   # direct + indirect path
    "collider (c)": (collider_world, DIRECT_EFFECT),
}

## 1. Adjust or not?

For each world: the true total effect, the average estimate **without** adjustment, and the average estimate **adjusting for** the third variable. The correct column should match the truth.

In [ ]:
rows = []
for name, (simulate, truth) in worlds.items():
    unadjusted, adjusted = [], []
    for _ in range(N_SIMS):
        y, x, third = simulate(N)
        unadjusted.append(coef_of_x(y, x))
        adjusted.append(coef_of_x(y, x, third))
    rows.append({"world": name, "true total effect": truth,
                 "unadjusted": np.mean(unadjusted), "adjusting for the third variable": np.mean(adjusted)})
results = pd.DataFrame(rows).set_index("world").round(2)
print(results)

ax = results.plot.bar(figsize=(8, 4), rot=0)
ax.set_ylabel("estimated effect of x on y")
plt.show()

How to read it:

- **Confounder:** only the adjusted estimate recovers the truth. Not adjusting mixes the effect of `z` into the estimate.
- **Mediator:** only the *unadjusted* estimate gives the **total** effect. Adjusting for `m` removes the part of the effect that travels through it, leaving only the direct effect. (If the direct effect is what you want, adjusting is right: the question defines the answer.)
- **Collider:** the unadjusted estimate is right. Adjusting for `c` creates an association that isn't there.

## 2. A confounder you didn't measure

In the confounder world, what if `z` is not in your dataset? We can only run the unadjusted analysis. Does collecting more data help?

In [ ]:
rows = []
for n in [50, 500, 5000, 50000]:
    estimates = [coef_of_x(*confounder_world(n)[:2]) for _ in range(100)]
    rows.append({"n": n, "mean estimate": np.mean(estimates), "SD of estimate": np.std(estimates)})
print(f"True effect: {DIRECT_EFFECT}")
print(pd.DataFrame(rows).set_index("n").round(3))

More data makes the estimate more precise (the SD shrinks), but it settles on the *wrong* value. A bigger sample can't fix a missing confounder, which is why the diagram must include variables you didn't measure and why the assumption has to be stated.

## Things to try

- Set `STRENGTH = 0`: all three worlds agree, because the third variable no longer matters. The bias grows with the strength of the connections.
- Set `STRENGTH = 2`: the unadjusted confounder estimate moves from 2.5 to 2.8, and adjusting for the collider now gives a negative effect (about -0.4) although the true effect is 2.0.
- Set `N = 100` and `N_SIMS = 1000`: the average estimates stay on the same values, but individual datasets scatter around them.
- Set `DIRECT_EFFECT = 0` (no effect of `x` on `y`): the collider world's unadjusted estimate is 0, while adjusting for `c` produces a spurious negative association. The confounder world's unadjusted estimate is non-zero for the same reason: an association, not an effect.